In [1]:
pip install --upgrade h3


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# --- 1. Librerías necesarias ---
import geopandas as gpd
from sqlalchemy import create_engine
import h3
from shapely.geometry import Polygon, shape
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
# --- 2. Conexión a PostGIS (usa las mismas variables del contenedor) ---
DB_USER = "postgres"
DB_PASS = "postgres"
DB_HOST = "db"         # nombre del servicio en docker-compose
DB_PORT = "5432"
DB_NAME = "agtech_db"
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [13]:
# --- 3. Leer la capa 'la_magdalena_L4' desde la base ---
gdf = gpd.read_postgis('SELECT * FROM "la_magdalena_L4"', con=engine, geom_col="geometry")

# Asignar CRS si no está definido
if gdf.crs is None:
    gdf.set_crs(epsg=4326, inplace=True)

print(f"✅ Capa cargada: {len(gdf)} features")
display(gdf.head(2))

✅ Capa cargada: 1 features


,name,folders,description,altitude,alt_mode,time_begin,time_end,time_when,extrude,tessellate,visibility,geometry
0,Lote 4,lote_4_perimetro,None,0.0,None,None,None,None,0,1,0,"MULTIPOLYGON Z (((-63.79861 -33.98198 0.00000,..."


In [1]:
from shapely.geometry import Polygon, mapping
import h3

# --- 4. Crear los hexágonos H3 con resolución 12 ---
poly = gdf.geometry.unary_union  
geojson_dict = mapping(poly)

# Generar índices H3 dentro del polígono (API moderna)
hex_ids = list(h3.polyfill(geojson_dict, res=12))

# Convertir los índices H3 en polígonos shapely
hex_geoms = [Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in hex_ids]

# Crear GeoDataFrame con atributos
hex_gdf = gpd.GeoDataFrame({
    "Id_Hexagono": hex_ids,
    "Empresa": "Garruchos agropecuaria",
    "Campaña": "2020–2023",
    "Campo": "La magdalena",
    "Lote": "L4",
    "Actividad": "Soja"
}, geometry=hex_geoms, crs="EPSG:4326")

print(f"✅ {len(hex_gdf)} hexágonos generados.")
display(hex_gdf.head(2))

NameError: name 'gdf' is not defined